<a href="https://colab.research.google.com/github/gauravd12345/ml_proj/blob/main/rnn/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re 
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

from nltk.tokenize import word_tokenize

In [2]:
with open("/Users/gauravd/Desktop/ml_proj/datasets/tiny_shakespeare/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

words = word_tokenize(text)

# vocab = "".join(sorted(set(text)))
vocab = sorted(set(words))
vocab_len = len(vocab)

ctoi, itoc = {}, {}
for i in range(len(vocab)):
    ctoi[vocab[i]] = i
    itoc[i] = vocab[i]

print(vocab_len)

14310


In [3]:
X, y = [], []
for i, j in zip(words, words[1:]):
    X.append(ctoi[i])
    y.append(ctoi[j])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(X[:10])
print(y[:10])

tensor([ 1152,   709,   223,   482, 13877, 10480,  3440,  7080,   219,  7604])
tensor([  709,   223,   482, 13877, 10480,  3440,  7080,   219,  7604,  8993])


In [4]:
"""
Input vector x(t) represents word in time t encoded using 1-of-N
coding and previous context layer - size of vector x is equal to
size of vocabulary V (this can be in practice 30 000 − 200 000)
plus size of context layer. Size of context (hidden) layer s is
usually 30 − 500 hidden units.

"""

hidden_size = 100
class RNN(nn.Module):
    def __init__(self):
        super().__init__() 
        self.x_t = nn.Embedding(vocab_len + hidden_size, hidden_size)
        self.s_t = self.x_t.weight.data[-100:] 
        self.y_t = nn.Linear(hidden_size, vocab_len)


    def forward(self, w):
        # x = torch.cat((w, self.s_t)) # shape is (V + H, )
        s = F.sigmoid(self.x_t(w))
        self.s_t = s
        y = F.softmax(self.y_t(s), dim=1)
        return y   
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNN().to(device)
print(f"device: {device}")

device: cpu


In [ ]:
lr = 0.1
epochs = 10

criterion = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

for epoch in range(epochs):
    optimizer.zero_grad()
    out = model(X.to(device))
    loss = criterion(out, y.to(device))

    loss.backward()
    optimizer.step()

    print(f"Loss: {loss:2f}")